# SCOPE xi(r) Investigation — §5.7 Shot Noise and Cosmic Variance

This notebook investigates the decomposition of the SCOPE estimator variance into its two primary components:
1. **Shot Noise:** The uncertainty arising from the finite number of galaxy pairs in the sub-volumes.
2. **Cosmic Variance (Sample Variance):** The uncertainty arising from the large-scale density fluctuations shared across sub-volumes and the finite volume of the simulation box.

We also examine how these components scale with N_subvol and how they differ between dense and sparse galaxy samples.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, str(Path('../../../../src').resolve()))
from config import get_snapshot_redshift

try:
    from utils.matplotlib_config import setconfig
    setconfig()
except ImportError:
    pass

DATA_ROOT = Path('../../../../data/2pcf/scope_xi')
MODEL     = 'lc16'
SIM       = 'L800'
N_REF     = 1024

In [ ]:
def load_raw_data(iz: int, mstar_tag: str, n_values: list[int]) -> pd.DataFrame:
    """Load per-seed CSVs for a set of N values."""
    base = DATA_ROOT / MODEL / f'iz{iz}' / mstar_tag
    all_dfs = []
    for n in n_values:
        files = sorted(base.glob(f'n{n}/seed*/scope_xi_{SIM}_iz{iz}.csv'))
        # Exclude seed 1000 (reference)
        files = [f for f in files if 'seed1000' not in str(f)]
        if files:
            all_dfs.append(pd.concat([pd.read_csv(f) for f in files], ignore_index=True))
    return pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

def compute_components(df: pd.DataFrame) -> pd.DataFrame:
    """Compute shot noise and total variance components per (n, bin)."""
    # Filter out rows where rr is zero to avoid division by zero
    df = df[df['rr'] > 0].copy()
    
    # Shot noise for SCOPE corrected estimator:
    # sigma_shot^2 = (alpha^2 * dd_auto + beta^2 * dd_cross) / rr^2
    df['sigma2_shot_corr'] = (df['alpha']**2 * df['dd_auto'] + df['beta']**2 * df['dd_cross']) / (df['rr']**2)
    
    # Aggregate over seeds
    # Use dd_auto/dd_cross > 0 to ensure we have some signal for variance?
    # Actually, we want to know variance even when signal is zero.
    stats = df.groupby(['n_subvol', 'bin_idx', 'r_mid']).agg(
        xi_corr_mean   = ('xi_corrected', 'mean'),
        xi_corr_var    = ('xi_corrected', 'var'),
        shot_noise_avg = ('sigma2_shot_corr', 'mean'),
        n_seeds        = ('selection_seed', 'count')
    ).reset_index()
    
    # Total variance is the variance across seeds
    stats['sigma_total'] = np.sqrt(stats['xi_corr_var'])
    
    # Shot noise component (averaged over seeds)
    stats['sigma_shot'] = np.sqrt(stats['shot_noise_avg'])
    
    # Cosmic Variance component:
    # sigma_total^2 = sigma_shot^2 + sigma_CV^2
    cv2 = stats['xi_corr_var'] - stats['shot_noise_avg']
    stats['sigma_cv'] = np.sqrt(np.maximum(cv2, 0))
    
    return stats


## 1. Dense Sample ($M_* > 10^9$)

We start with the dense $M_* > 10^9$ selection at $z=1.5$ (iz155).

In [ ]:
IZ = 155
N_VALUES = [8, 32, 128]
MSTAR = 'mstar9.0'

df_9 = load_raw_data(IZ, MSTAR, N_VALUES)
stats_9 = compute_components(df_9)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, n in enumerate(N_VALUES):
    ax = axes[i]
    sub = stats_9[stats_9['n_subvol'] == n].sort_values('r_mid')
    
    ax.plot(sub['r_mid'], sub['sigma_total'], 'k-', lw=2, label='Total Variance')
    ax.plot(sub['r_mid'], sub['sigma_shot'], 'C0--', lw=1.5, label='Shot Noise')
    ax.plot(sub['r_mid'], sub['sigma_cv'], 'C1:', lw=1.5, label='Cosmic Variance')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'$N_{{\\rm subvol}} = {n}$')
    ax.set_xlabel(r'$r$ [$h^{-1}$Mpc]')
    if i == 0:
        ax.set_ylabel(r'$\sigma_{\xi}$')
        ax.legend()

fig.suptitle(f'Variance Decomposition — {MSTAR} (iz{IZ})', fontsize=14)
    # Filter for positive variance to avoid log scale issues
    sub = sub[sub['sigma_total'] > 0]
    if sub.empty: continue
    
    ax.plot(sub['r_mid'], sub['sigma_total'], 'k-', lw=2, label='Total Variance')
    ax.plot(sub['r_mid'], sub['sigma_shot'], 'C0--', lw=1.5, label='Shot Noise')
    ax.plot(sub['r_mid'], sub['sigma_cv'], 'C1:', lw=1.5, label='Cosmic Variance')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'$N_{{\\rm subvol}} = {n}$')
    ax.set_xlabel(r'$r$ [$h^{-1}$Mpc]')
    if i == 0:
        ax.set_ylabel(r'$\sigma_{\xi}$')
        ax.legend()

fig.suptitle(f'Variance Decomposition — {MSTAR if cell["id"] == "plot-9.0" else "MSTAR_SPARSE"} (iz{IZ})', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Sparse Sample ($M_* > 10^{11}$)

In sparse samples, shot noise should dominate even at relatively small scales.

*Note: For the very sparse $M_* > 10^{11}$ selection, most separation bins in individual sub-volumes contain zero galaxy pairs at small $N_{\rm subvol}$. In these cases, the variance is calculated over identical -1.0 values, yielding zero plottable variance on a log scale. Non-zero variance only appears at very large scales where pair counts are non-zero.*

In [ ]:
MSTAR_SPARSE = 'mstar11.0'
df_11 = load_raw_data(IZ, MSTAR_SPARSE, N_VALUES)
stats_11 = compute_components(df_11)

fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)

for i, n in enumerate(N_VALUES):
    ax = axes[i]
    sub = stats_11[stats_11['n_subvol'] == n].sort_values('r_mid')
    
    ax.plot(sub['r_mid'], sub['sigma_total'], 'k-', lw=2, label='Total Variance')
    ax.plot(sub['r_mid'], sub['sigma_shot'], 'C0--', lw=1.5, label='Shot Noise')
    ax.plot(sub['r_mid'], sub['sigma_cv'], 'C1:', lw=1.5, label='Cosmic Variance')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'$N_{{\\rm subvol}} = {n}$')
    ax.set_xlabel(r'$r$ [$h^{-1}$Mpc]')
    if i == 0:
        ax.set_ylabel(r'$\sigma_{\xi}$')
        ax.legend()

fig.suptitle(f'Variance Decomposition — {MSTAR_SPARSE} (iz{IZ})', fontsize=14)
    # Filter for positive variance to avoid log scale issues
    sub = sub[sub['sigma_total'] > 0]
    if sub.empty: continue
    
    ax.plot(sub['r_mid'], sub['sigma_total'], 'k-', lw=2, label='Total Variance')
    ax.plot(sub['r_mid'], sub['sigma_shot'], 'C0--', lw=1.5, label='Shot Noise')
    ax.plot(sub['r_mid'], sub['sigma_cv'], 'C1:', lw=1.5, label='Cosmic Variance')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'$N_{{\\rm subvol}} = {n}$')
    ax.set_xlabel(r'$r$ [$h^{-1}$Mpc]')
    if i == 0:
        ax.set_ylabel(r'$\sigma_{\xi}$')
        ax.legend()

fig.suptitle(f'Variance Decomposition — {MSTAR if cell["id"] == "plot-9.0" else "MSTAR_SPARSE"} (iz{IZ})', fontsize=14)
plt.tight_layout()
plt.show()

## 3. Scaling with N_subvol

We examine how each component scales with the number of sub-volumes at a fixed large-scale separation (r ≈ 20 h^-1 Mpc).

In [ ]:
R_TARGET = 20.0
N_FULL = [4, 8, 16, 32, 64, 128, 256]

def get_scaling_data(mtag):
    df = load_raw_data(IZ, mtag, N_FULL)
    stats = compute_components(df)
    
    # Find bin closest to R_TARGET
    bins = stats['bin_idx'].unique()
    bin_r = stats[stats['n_subvol'] == stats['n_subvol'].max()].sort_values('bin_idx')
    best_bin = bin_r.iloc[(bin_r['r_mid'] - R_TARGET).abs().argsort()[:1]]['bin_idx'].values[0]
    
    scaling = stats[stats['bin_idx'] == best_bin].sort_values('n_subvol')
    return scaling

scaling_9 = get_scaling_data('mstar9.0')
scaling_11 = get_scaling_data('mstar11.0')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, data, mtag in zip(axes, [scaling_9, scaling_11], ['mstar9.0', 'mstar11.0']):
    ax.plot(data['n_subvol'], data['sigma_total'], 'ko-', label='Total')
    ax.plot(data['n_subvol'], data['sigma_shot'], 'C0s--', label='Shot Noise')
    ax.plot(data['n_subvol'], data['sigma_cv'], 'C1d:', label='Cosmic Variance')
    
    # 1/sqrt(N) reference
    n_ref = data['n_subvol'].values
    y_ref = data['sigma_total'].iloc[0] * np.sqrt(data['n_subvol'].iloc[0] / n_ref)
    ax.plot(n_ref, y_ref, 'grey', lw=1, ls='-', label=r'$1/\sqrt{N}$ expected')
    
    ax.set_xscale('log')
    ax.set_yscale('log')
    ax.set_title(f'Scaling at r ≈ {R_TARGET} h^-1 Mpc ({mtag})')
    ax.set_xlabel(r'$N_{\rm subvol}$')
    ax.set_ylabel(r'$\sigma$')
    ax.legend()

plt.tight_layout()
plt.show()